# Notebook 21 — Candidate Generation (Wikidata) v4

**Design reference:** `documentation/Wikidata/2026-04-26_investigation/13_architecture_design.md`  
**Rules reference:** `documentation/Wikidata/2026-04-26_investigation/08_known_rules.md`  
**v3 archive:** `21_candidate_generation_wikidata_v3_archive.ipynb`

This notebook is the v4 implementation. It is intentionally empty until Stage 5 implementation begins.  
All design decisions are in the architecture document above — do not derive design from the archived v3 notebook.

## Step 1 — Setup

Load config and initialize shared state.

In [ ]:
import sys
from pathlib import Path

# Resolve repo root (two levels up from speakermining/src/process/notebooks/)
_nb_path = Path().resolve()
repo_root = _nb_path
for _ in range(6):
    if (repo_root / "speakermining").exists():
        break
    repo_root = repo_root.parent

src_root = repo_root / "speakermining" / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

# print(f"repo_root: {repo_root}")
# print(f"src_root:  {src_root}")


In [ ]:
from process.candidate_generation.wikidata.config import load_config

cfg = load_config(repo_root)
print(f"max_queries_per_run:        {cfg.max_queries_per_run}")
print(f"depth_limit:                {cfg.depth_limit}")
print(f"class_hierarchy_depth_limit:{cfg.class_hierarchy_depth_limit}")
print(f"deferred_basic_fetch_mode:  {cfg.deferred_basic_fetch_mode}")
print(f"label_languages:            {cfg.label_languages}")


## Step 2 — Run ExternalEventReaders

Register seeds, core classes, and rules into the event store (idempotent).

In [ ]:
from process.candidate_generation.wikidata.external_readers.seed_reader import SeedReader
from process.candidate_generation.wikidata.external_readers.core_class_reader import CoreClassReader
from process.candidate_generation.wikidata.external_readers.relevancy_rule_reader import RelevancyRuleReader
from process.candidate_generation.wikidata.external_readers.full_fetch_rule_reader import FullFetchRuleReader
from process.candidate_generation.wikidata.external_readers.rewire_catalogue_reader import RewireCatalogueReader

readers = [
    SeedReader(repo_root),
    CoreClassReader(repo_root),
    RelevancyRuleReader(repo_root),
    FullFetchRuleReader(repo_root),
    RewireCatalogueReader(repo_root),
]
for reader in readers:
    n = reader.run()
    print(f"{type(reader).__name__}: {n} events emitted")

## Step 3 — Initialize Handlers and Replay Event Store

Each handler replays from its last processed sequence. After replay, all work queues reflect the current event log state.

In [ ]:
from process.candidate_generation.wikidata.event_writer import get_event_store
from process.candidate_generation.wikidata.handlers.seed_handler import SeedHandler
from process.candidate_generation.wikidata.handlers.class_hierarchy_handler import ClassHierarchyHandler
from process.candidate_generation.wikidata.handlers.relevancy_handler import RelevancyHandler
from process.candidate_generation.wikidata.handlers.fetch_decision_handler import FetchDecisionHandler
from process.candidate_generation.wikidata.handlers.basic_fetch_handler import BasicFetchHandler
from process.candidate_generation.wikidata.handlers.full_fetch_handler import FullFetchHandler
from process.candidate_generation.wikidata.handlers.entity_lookup_handler import EntityLookupIndexHandler
from process.candidate_generation.wikidata.handlers.output_handler import CoreClassOutputHandler

store = get_event_store(repo_root)

seed_h        = SeedHandler(repo_root, store)
class_h       = ClassHierarchyHandler(repo_root, store, depth_limit=cfg.class_hierarchy_depth_limit)
relevancy_h   = RelevancyHandler(repo_root, store)
fetch_dec_h   = FetchDecisionHandler(repo_root, store)
basic_fetch_h = BasicFetchHandler(repo_root, store)
full_fetch_h  = FullFetchHandler(repo_root, store, depth_limit=cfg.depth_limit)
lookup_h      = EntityLookupIndexHandler(repo_root, store)
output_h      = CoreClassOutputHandler(repo_root, store)

# Handler order enforces processing dependencies (ClassHierarchy → Relevancy → FetchDecision → ...)
_all_handlers = [seed_h, class_h, relevancy_h, fetch_dec_h, basic_fetch_h, full_fetch_h, lookup_h, output_h]

def replay_all():
    for h in _all_handlers:
        h.replay()

replay_all()
print("Replay complete.")
print(f"  Seeds registered:   {len(seed_h.seeds())}")
print(f"  Classes resolved:   {len(class_h._resolved)}")
print(f"  Entities relevant:  {len(relevancy_h._relevant)}")
print(f"  Full-fetch pending: {len(full_fetch_h._queue)}")


## Step 4 — Initialize HTTP Request Context

Required before any network calls. Sets budget from config.

In [ ]:
import os
from process.candidate_generation.wikidata.cache import begin_request_context

# Wire adaptive backoff parameters via environment variables (read by heartbeat_monitor.py)
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_ENABLED"] = "1" if cfg.adaptive_backoff_enabled else "0"
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_PATTERN_HEARTBEATS"] = str(cfg.adaptive_backoff_pattern_heartbeats)
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_INCREASE_FACTOR"] = str(cfg.adaptive_backoff_increase_factor)
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_DECREASE_FACTOR"] = str(cfg.adaptive_backoff_decrease_factor)
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_MIN_DELAY_SECONDS"] = str(cfg.adaptive_backoff_min_delay_seconds)
os.environ["WIKIDATA_ADAPTIVE_BACKOFF_MAX_DELAY_SECONDS"] = str(cfg.adaptive_backoff_max_delay_seconds)

begin_request_context(
    budget_remaining=cfg.max_queries_per_run,
    query_delay_seconds=cfg.query_delay_seconds,
    progress_every_calls=cfg.progress_every_calls,
    progress_every_seconds=cfg.progress_every_seconds,
    http_max_retries=cfg.http_max_retries,
    http_backoff_base_seconds=cfg.http_backoff_base_seconds,
    context_label="wikidata_v4",
)
print(f"Request context initialized.")
print(f"  Budget:        {cfg.max_queries_per_run} queries")
print(f"  Query delay:   {cfg.query_delay_seconds}s  (adaptive: {cfg.adaptive_backoff_min_delay_seconds}s – {cfg.adaptive_backoff_max_delay_seconds}s)")
print(f"  Adaptive:      {'on' if cfg.adaptive_backoff_enabled else 'off'}")


## Step 5 — Main Work Loop

Priority order: ClassHierarchy → BasicFetch (immediate) → FullFetch → repeat.

`run_with_progress_heartbeat` provides:
- **Heartbeat thread** — prints status every `progress_every_seconds` seconds (default 60s)
- **Network call progress** — already built into `cache._http_get_json` every `progress_every_calls` calls (default 50)
- **Graceful shutdown** — Ctrl+C sets a flag; the active handler finishes its current save point before the loop exits cleanly
- **Adaptive backoff** — adjusts `query_delay_seconds` at each heartbeat based on observed 429/503 patterns

In [ ]:
from process.candidate_generation.wikidata.heartbeat_monitor import run_with_progress_heartbeat
from process.candidate_generation.wikidata.graceful_shutdown import should_terminate, reset_termination_flag

_langs = cfg.label_languages

def _work_loop():
    iterations = 0
    while not should_terminate():
        # Priority 1: drain class hierarchy resolution queue
        if class_h.has_pending():
            if class_h.resolve_next(languages=_langs) > 0:
                replay_all()
            iterations += 1
            continue

        # Priority 2: drain immediate basic_fetch queue (batch)
        if basic_fetch_h.has_immediate_pending():
            if basic_fetch_h.do_next_batch(languages=_langs) > 0:
                replay_all()
            iterations += 1
            continue

        # Priority 3: process next full_fetch
        if full_fetch_h.has_pending():
            if full_fetch_h.do_next(languages=_langs) > 0:
                replay_all()
            iterations += 1
            continue

        break  # all primary queues empty

    print(f"Main loop complete after {iterations} iterations.")
    print(f"  Classes resolved:   {len(class_h._resolved)}")
    print(f"  Entities relevant:  {len(relevancy_h._relevant)}")
    print(f"  Full-fetched:       {len(full_fetch_h._done)}")

reset_termination_flag()
run_with_progress_heartbeat(
    repo_root,
    phase="main_loop",
    work_label="v4 fetch loop",
    work_fn=_work_loop,
    heartbeat_interval_seconds=int(cfg.progress_every_seconds),
    heartbeat_window_size=25,
)


## Step 6 — Deferred BasicFetch (optional)

Only runs if `deferred_basic_fetch_mode = "end_of_run"` in config.

In [ ]:
if cfg.deferred_basic_fetch_mode == "end_of_run":
    _deferred = 0
    while basic_fetch_h.has_deferred_pending() and not should_terminate():
        basic_fetch_h.do_next_deferred_batch(languages=_langs)
        replay_all()
        _deferred += 1
    print(f"Deferred basic_fetch: {_deferred} batches processed.")
else:
    print(f"Deferred basic_fetch skipped (mode={cfg.deferred_basic_fetch_mode!r}).")


## Step 7 — Write Output Files

Write `core_<class>.json` and `not_relevant_core_<class>.json` for all core classes.

In [ ]:
from process.candidate_generation.wikidata.schemas import build_artifact_paths

paths = build_artifact_paths(repo_root)
output_h._write(paths.projections_dir)

for core_qid, label in sorted(output_h._core_classes.items()):
    from process.candidate_generation.wikidata.schemas import canonical_class_filename
    try:
        fname = canonical_class_filename(label)
    except ValueError:
        fname = core_qid.lower()
    rel_file = paths.projections_dir / f"core_{fname}.json"
    not_rel_file = paths.projections_dir / f"not_relevant_core_{fname}.json"
    import json
    rel_count = len(json.loads(rel_file.read_text(encoding="utf-8"))) if rel_file.exists() else 0
    not_rel_count = len(json.loads(not_rel_file.read_text(encoding="utf-8"))) if not_rel_file.exists() else 0
    print(f"  {label} ({core_qid}): {rel_count} relevant, {not_rel_count} not-relevant")

print("Output files written.")
